# M3-TTS: Phase 1 — Khmer TTS Dataset Setup
## Google Colab Notebook

This notebook validates the Khmer TSV dataset, audio files, and verifies the StyleTTS2 training framework.

**Phase 1 does NOT perform full training.**

> Every code cell is written to fail gracefully. If a required variable or file
> is missing, the cell prints a clear message and skips its work instead of
> crashing with `NameError` / `TypeError`.

## 1. Configuration

Edit the column mappings below to match your TSV header.

`DATASET_DIR`, `AUDIO_DIR` and `METADATA_TSV` are populated automatically after
unzipping (Section 4/5) — do not edit them manually.

In [ ]:
# =============================================================================
# CONFIGURATION — Edit these to match your dataset
# =============================================================================

AUDIO_COLUMN = "audio"      # Column name for audio file paths
TEXT_COLUMN = "text"        # Column name for Khmer transcripts
SPEAKER_COLUMN = "speaker"  # Column name for speaker ID

# ZIP file location on Google Drive (at root level)
ZIP_FILENAME = "khmer_tts_data.zip"
ZIP_PATH = f"/content/drive/MyDrive/{ZIP_FILENAME}"

# Working directories
EXTRACT_DIR = "/content/khmer_tts_data"          # Where zip is extracted (Colab local)
PROJECT_ROOT = "/content/drive/MyDrive/khmer_tts"  # Google Drive output

# DATASET_DIR, AUDIO_DIR, METADATA_TSV are set automatically after unzip
# in Sections 4 & 5. Do NOT edit them manually.
DATASET_DIR = None
AUDIO_DIR = None
METADATA_TSV = None

# Output directories (on Google Drive)
PROCESSED_DIR = f"{PROJECT_ROOT}/processed_dataset"
REPORTS_DIR = f"{PROJECT_ROOT}/reports"
CHECKPOINTS_DIR = f"{PROJECT_ROOT}/checkpoints"
SAMPLES_DIR = f"{PROJECT_ROOT}/samples"
TEST_DIR = f"{PROCESSED_DIR}/test"

# Audio settings
TARGET_SAMPLE_RATE = 22050
TARGET_CHANNELS = 1
MIN_DURATION_SEC = 0.5
MAX_DURATION_SEC = 30.0

# DataFrame placeholder — populated in Section 7
df = None

print("Configuration loaded.")
print(f"  Audio column:    {AUDIO_COLUMN}")
print(f"  Text column:     {TEXT_COLUMN}")
print(f"  Speaker column:  {SPEAKER_COLUMN}")
print(f"  ZIP file:        {ZIP_PATH}")


## 2. Environment Detection

Reports Python, PyTorch / CUDA and GPU info. Uses `getattr` for `total_memory`
so it still works when that attribute is unavailable.

In [ ]:
import sys
import os
import platform

print("=" * 60)
print("ENVIRONMENT DETECTION")
print("=" * 60)

# Python version
print(f"Python:         {sys.version.splitlines()[0]}")
print(f"Platform:       {platform.platform()}")

# PyTorch + CUDA
try:
    import torch
    print(f"PyTorch:        {torch.__version__}")
    if hasattr(torch, "cuda") and torch.cuda.is_available():
        print(f"CUDA Available: True")
        try:
            print(f"CUDA Version:   {torch.version.cuda}")
        except Exception:
            print("CUDA Version:   unknown")
        try:
            gpu_name = torch.cuda.get_device_name(0)
            print(f"GPU:            {gpu_name}")
        except Exception:
            gpu_name = "unknown"
            print("GPU:            unknown")
        try:
            props = torch.cuda.get_device_properties(0)
            gpu_mem = getattr(props, "total_memory", 0) / (1024 ** 3)
            print(f"VRAM:           {gpu_mem:.1f} GB")
        except Exception:
            print("VRAM:           unknown")
    else:
        print("CUDA Available: False")
        print("GPU:            None detected")
except ImportError:
    print("PyTorch:        NOT INSTALLED")
except Exception as e:
    print(f"PyTorch probe error: {e}")

print("=" * 60)


## 3. Mount Google Drive

Skips mounting if Google Drive is already mounted.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Not running in Google Colab — 'google.colab' not available.")
    print("Skipping Google Drive mount. Local paths will be used as-is.")
else:
    try:
        if os.path.exists("/content/drive/MyDrive"):
            print("Google Drive already mounted at /content/drive")
        else:
            drive.mount("/content/drive")
            print("Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Google Drive mount failed: {e}")


## 4. Unzip Dataset

Finds `khmer_tts_data.zip` (or any `.zip`) in MyDrive recursively, then extracts
it to `/content/khmer_tts_data`. If nothing is found it lists MyDrive contents.

In [ ]:
import zipfile
import glob

# ---------------------------------------------------------------------------
# Defensive: ensure configuration variables exist
# ---------------------------------------------------------------------------
try:
    ZIP_PATH
except NameError:
    ZIP_PATH = "/content/drive/MyDrive/khmer_tts_data.zip"
try:
    EXTRACT_DIR
except NameError:
    EXTRACT_DIR = "/content/khmer_tts_data"
try:
    PROCESSED_DIR
except NameError:
    PROCESSED_DIR = "/content/drive/MyDrive/khmer_tts/processed_dataset"
try:
    REPORTS_DIR
except NameError:
    REPORTS_DIR = "/content/drive/MyDrive/khmer_tts/reports"
try:
    CHECKPOINTS_DIR
except NameError:
    CHECKPOINTS_DIR = "/content/drive/MyDrive/khmer_tts/checkpoints"
try:
    SAMPLES_DIR
except NameError:
    SAMPLES_DIR = "/content/drive/MyDrive/khmer_tts/samples"
try:
    TEST_DIR
except NameError:
    TEST_DIR = f"{PROCESSED_DIR}/test"

# Create output dirs (best effort)
for d in [PROCESSED_DIR, REPORTS_DIR, CHECKPOINTS_DIR, SAMPLES_DIR, TEST_DIR]:
    try:
        os.makedirs(d, exist_ok=True)
    except Exception as e:
        print(f"WARNING: could not create {d}: {e}")

# ---------------------------------------------------------------------------
# Find the zip file
# ---------------------------------------------------------------------------
print(f"Looking for: {ZIP_PATH}")

if not os.path.exists(ZIP_PATH):
    candidates = []
    try:
        if os.path.exists("/content/drive/MyDrive"):
            candidates = sorted(set(
                glob.glob("/content/drive/MyDrive/*.zip") +
                glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True)
            ))
    except Exception as e:
        print(f"  Search error: {e}")

    print(f"  All zips found in Drive: {candidates}")
    if candidates:
        khmer_zips = [c for c in candidates if "khmer" in c.lower() or "tts" in c.lower()]
        ZIP_PATH = khmer_zips[0] if khmer_zips else candidates[0]
        print(f"  Using: {ZIP_PATH}")
    else:
        print("  No zip files found in MyDrive.")
        print("  Contents of MyDrive root:")
        try:
            if os.path.exists("/content/drive/MyDrive"):
                for item in sorted(os.listdir("/content/drive/MyDrive")):
                    print(f"    {item}")
            else:
                print("    /content/drive/MyDrive does not exist (Drive not mounted?).")
        except Exception as e:
            print(f"    Could not list MyDrive: {e}")

# ---------------------------------------------------------------------------
# Unzip
# ---------------------------------------------------------------------------
if os.path.exists(ZIP_PATH):
    try:
        already = os.path.exists(EXTRACT_DIR) and bool(os.listdir(EXTRACT_DIR))
    except Exception:
        already = False

    if not already:
        print(f"\nExtracting {ZIP_PATH} ...")
        try:
            os.makedirs(EXTRACT_DIR, exist_ok=True)
            with zipfile.ZipFile(ZIP_PATH, "r") as z:
                z.extractall(EXTRACT_DIR)
            print(f"  Extracted to: {EXTRACT_DIR}")
        except Exception as e:
            print(f"  ERROR extracting zip: {e}")
    else:
        print(f"  Already extracted at: {EXTRACT_DIR}")

    print("\nExtracted contents (top 2 levels):")
    try:
        for root, dirs_list, files in os.walk(EXTRACT_DIR):
            level = root.replace(EXTRACT_DIR, "").count(os.sep)
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")
            if level < 2:
                for f in files[:10]:
                    print(f"{indent}  {f}")
                if len(files) > 10:
                    print(f"{indent}  ... and {len(files) - 10} more files")
    except Exception as e:
        print(f"  Could not walk extracted dir: {e}")
else:
    print("\nERROR: No zip file found. Please upload it to MyDrive/ root.")


## 5. Detect Dataset Paths

Walks the extracted directory to locate a `.tsv` metadata file and the audio
folder. Sets `METADATA_TSV`, `AUDIO_DIR` and `DATASET_DIR`.

In [ ]:
# Defensive: ensure variables exist
for _var, _default in [('EXTRACT_DIR', '/content/khmer_tts_data'), ('METADATA_TSV', None), ('AUDIO_DIR', None)]:
    try:
        eval(_var)
    except NameError:
        globals()[_var] = _default

def find_dataset_paths(root_dir):
    metadata_files = []
    audio_dirs = []
    all_extensions = set()
    try:
        for dirpath, dirnames, filenames in os.walk(root_dir):
            for f in filenames:
                ext = os.path.splitext(f)[1].lower()
                all_extensions.add(ext)
                if ext in ('.tsv', '.csv', '.txt', '.json'):
                    metadata_files.append(os.path.join(dirpath, f))
            wav_count = sum(1 for f in filenames if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a')))
            if wav_count > 0:
                audio_dirs.append((dirpath, wav_count))
    except Exception as e:
        print(f'Walk error: {e}')

    metadata_file = None
    for t in metadata_files:
        if 'metadata' in os.path.basename(t).lower():
            metadata_file = t
            break
    if not metadata_file and metadata_files:
        metadata_file = metadata_files[0]

    audio_dir = max(audio_dirs, key=lambda x: x[1])[0] if audio_dirs else None
    return metadata_file, audio_dir, metadata_files, all_extensions

if not os.path.exists(EXTRACT_DIR) or not os.listdir(EXTRACT_DIR):
    print(f'ERROR: EXTRACT_DIR empty: {EXTRACT_DIR}')
    print('Run Section 4 (Unzip) first.')
else:
    metadata_file, audio_dir, meta_files, extensions = find_dataset_paths(EXTRACT_DIR)
    print(f'Extensions found: {sorted(extensions)}')
    print(f'Metadata candidates: {meta_files[:10]}')

    if metadata_file:
        METADATA_TSV = metadata_file
        print(f'Metadata found: {METADATA_TSV}')

    if audio_dir:
        AUDIO_DIR = audio_dir
        wav_count = sum(1 for f in os.listdir(audio_dir) if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a')))
        print(f'Audio dir: {AUDIO_DIR} ({wav_count} files)')

    if METADATA_TSV and AUDIO_DIR:
        DATASET_DIR = os.path.dirname(METADATA_TSV)
        print('\nDataset paths auto-detected. Ready.')
    elif AUDIO_DIR and not METADATA_TSV:
        print('\n' + '='*60)
        print('NO METADATA FILE FOUND - Generating from filenames')
        print('='*60)
        sample_files = sorted(os.listdir(AUDIO_DIR))[:10]
        print(f'\nSample filenames ({AUDIO_DIR}):')
        for f in sample_files:
            print(f'  {f}')
        print(f'\nTotal audio files: {wav_count}')

        try:
            import pandas as pd
            audio_files = sorted([f for f in os.listdir(AUDIO_DIR)
                                  if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a'))])

            base = os.path.splitext(audio_files[0])[0]
            parts = base.replace('_', ' ').replace('-', ' ').split()
            print(f'\nFirst filename base: {base}')
            print(f'Parts: {parts}')
            print(f'Number of parts: {len(parts)}')

            print('\nAttempting auto-generation of metadata.tsv from filenames...')
            data = []
            for af in audio_files:
                name_no_ext = os.path.splitext(af)[0]
                text_part = name_no_ext.replace('_', ' ').replace('-', ' ')
                data.append({'audio': af, 'text': text_part, 'speaker': 'default'})

            df = pd.DataFrame(data)
            auto_tsv = os.path.join(os.path.dirname(AUDIO_DIR), 'metadata.tsv')
            df.to_csv(auto_tsv, sep='\t', index=False)
            METADATA_TSV = auto_tsv
            DATASET_DIR = os.path.dirname(auto_tsv)
            print(f'\nCreated: {auto_tsv} ({len(df)} rows)')
            print('WARNING: Text column contains filename-derived text, NOT actual Khmer transcripts.')
            print('Replace metadata.tsv with your real transcript file for training.')
        except Exception as e:
            print(f'Auto-generation failed: {e}')
            print('\nPlease upload metadata.tsv to the dataset directory.')

        if METADATA_TSV and AUDIO_DIR:
            print('\nProceeding with auto-generated metadata. Ready.')


## 6. Install Dependencies

Installs and imports the Python packages used throughout the notebook.

In [ ]:
import subprocess

def pip_install(pkg):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       check=False, capture_output=True, text=True)
        return True
    except Exception as e:
        print(f"  pip install {pkg} failed: {e}")
        return False

for pkg in ["pandas", "numpy", "librosa", "soundfile", "scipy", "tqdm", "matplotlib"]:
    print(f"Installing {pkg} ...")
    pip_install(pkg)

import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from scipy import signal
from pathlib import Path
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings("ignore")

print("Dependencies imported.")


## 7. Load and Inspect TSV

Loads the metadata TSV with `pandas.read_csv(sep="\t")`. Skips if
`METADATA_TSV` is not set or the file is missing.

In [ ]:
# Defensive: ensure key variables exist
try:
    METADATA_TSV
except NameError:
    METADATA_TSV = None
try:
    df
except NameError:
    df = None

if METADATA_TSV is None:
    print('ERROR: METADATA_TSV is not set. Run Sections 4 & 5 first.')
elif not isinstance(METADATA_TSV, str) or not os.path.exists(METADATA_TSV):
    print(f'ERROR: Metadata not found at {METADATA_TSV}')
else:
    try:
        print(f'File: {METADATA_TSV}')
        print(f'Size: {os.path.getsize(METADATA_TSV)} bytes')
        print(f'\nFirst 20 lines (raw):')
        with open(METADATA_TSV, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= 20:
                    break
                print(f'  [{i}] {repr(line)}')

        print(f'\nAttempting to detect separator...')
        with open(METADATA_TSV, 'r', encoding='utf-8') as f:
            first_lines = [f.readline() for _ in range(5)]
        for sep_name, sep in [('tab', '\\t'), ('comma', ','), ('pipe', '|'), ('semicolon', ';')]:
            counts = [line.count(sep) for line in first_lines]
            print(f'  {sep_name}: {counts}')

        print(f'\nTrying to load with different separators...')
        for sep_name, sep in [('tab', '\\t'), ('comma', ','), ('pipe', '|'), ('semicolon', ';')]:
            try:
                test_df = pd.read_csv(METADATA_TSV, sep=sep, nrows=5, dtype=str)
                if len(test_df.columns) > 1:
                    print(f'  SUCCESS with {sep_name} separator!')
                    print(f'  Columns: {list(test_df.columns)}')
                    print(f'  First row: {dict(test_df.iloc[0])}')
                    df = pd.read_csv(METADATA_TSV, sep=sep, dtype=str)
                    print(f'  Loaded {len(df)} rows')
                    break
            except Exception as e:
                print(f'  Failed with {sep_name}: {str(e)[:80]}')
        else:
            print('\nCould not auto-detect format. Showing raw content above.')
    except Exception as e:
        print(f'ERROR: {e}')
        df = None


## 8. Validate Column Mapping

Confirms that the configured column names exist in the loaded dataframe.

In [ ]:
# Defensive defaults
try:
    AUDIO_COLUMN
except NameError:
    AUDIO_COLUMN = "audio"
try:
    TEXT_COLUMN
except NameError:
    TEXT_COLUMN = "text"
try:
    SPEAKER_COLUMN
except NameError:
    SPEAKER_COLUMN = "speaker"
try:
    df
except NameError:
    df = None

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
else:
    try:
        available_cols = list(df.columns)
        print(f"Available columns: {available_cols}")

        missing = []
        for col_name, col_val in [
            ("audio", AUDIO_COLUMN),
            ("text", TEXT_COLUMN),
            ("speaker", SPEAKER_COLUMN),
        ]:
            if col_val in available_cols:
                print(f"  [OK] {col_name} -> '{col_val}'")
            else:
                print(f"  [MISSING] {col_name} column '{col_val}' not found!")
                missing.append(col_val)

        if missing:
            print(f"\nWARNING: Missing columns: {missing}")
            print("Please update AUDIO_COLUMN, TEXT_COLUMN, SPEAKER_COLUMN at the top.")
        else:
            print("\nAll column mappings valid.")
    except Exception as e:
        print(f"ERROR validating column mapping: {e}")


## 9. TSV Validation

Checks missing values, empty transcripts, duplicate audio paths and duplicate
transcripts, then writes a text report.

In [ ]:
# Defensive defaults
try:
    df
except NameError:
    df = None
try:
    METADATA_TSV
except NameError:
    METADATA_TSV = None
try:
    AUDIO_COLUMN
except NameError:
    AUDIO_COLUMN = "audio"
try:
    TEXT_COLUMN
except NameError:
    TEXT_COLUMN = "text"
try:
    REPORTS_DIR
except NameError:
    REPORTS_DIR = "/content/drive/MyDrive/khmer_tts/reports"

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
else:
    try:
        report_lines = []
        report_lines.append("TSV VALIDATION REPORT")
        report_lines.append("=" * 60)
        report_lines.append(f"File: {METADATA_TSV}")
        report_lines.append(f"Total rows: {len(df)}")
        report_lines.append(f"Columns: {list(df.columns)}")
        report_lines.append("")

        missing_per_col = df.isnull().sum()
        report_lines.append("MISSING VALUES:")
        for col in df.columns:
            report_lines.append(f"  {col}: {missing_per_col[col]}")
        report_lines.append("")

        if TEXT_COLUMN in df.columns:
            empty_text = df[df[TEXT_COLUMN].isnull() | (df[TEXT_COLUMN].astype(str).str.strip() == "")]
            report_lines.append(f"Empty transcripts: {len(empty_text)}")
        else:
            empty_text = pd.DataFrame()
            report_lines.append("Text column not found - cannot check empty transcripts")
        report_lines.append("")

        if AUDIO_COLUMN in df.columns:
            missing_audio = df[df[AUDIO_COLUMN].isnull() | (df[AUDIO_COLUMN].astype(str).str.strip() == "")]
            report_lines.append(f"Missing audio paths: {len(missing_audio)}")
        else:
            missing_audio = pd.DataFrame()
            report_lines.append("Audio column not found - cannot check missing audio paths")
        report_lines.append("")

        if AUDIO_COLUMN in df.columns:
            dup_audio = df[df.duplicated(subset=[AUDIO_COLUMN], keep=False)]
            report_lines.append(f"Duplicate audio paths: {len(dup_audio)} rows")
        else:
            dup_audio = pd.DataFrame()
            report_lines.append("Audio column not found - cannot check duplicate audio paths")
        report_lines.append("")

        if TEXT_COLUMN in df.columns:
            dup_text = df[df.duplicated(subset=[TEXT_COLUMN], keep=False)]
            report_lines.append(f"Duplicate transcripts: {len(dup_text)} rows")
            dup_ratio = len(dup_text) / len(df) if len(df) > 0 else 0
            if dup_ratio > 0.3:
                report_lines.append(f"  WARNING: {dup_ratio:.1%} of rows have duplicate text")
        else:
            dup_text = pd.DataFrame()
            report_lines.append("Text column not found - cannot check duplicate transcripts")
        report_lines.append("")

        critical_cols = [c for c in [AUDIO_COLUMN, TEXT_COLUMN] if c in df.columns]
        if critical_cols:
            mask = df[critical_cols].isnull()
            mask = mask | (df[critical_cols].astype(str).apply(lambda s: s.str.strip() == ""))
            invalid_rows = df[mask.any(axis=1)]
            report_lines.append(f"Invalid rows (missing critical data): {len(invalid_rows)}")
        else:
            invalid_rows = pd.DataFrame()
            report_lines.append("No critical columns found to check invalid rows")
        report_lines.append("")

        report_text = "\n".join(report_lines)
        print(report_text)

        try:
            os.makedirs(REPORTS_DIR, exist_ok=True)
            with open(f"{REPORTS_DIR}/tsv_validation_report.txt", "w", encoding="utf-8") as f:
                f.write(report_text)

            invalid_all = pd.concat([empty_text, missing_audio, dup_audio, dup_text, invalid_rows]).drop_duplicates()
            if len(invalid_all) > 0:
                invalid_all.to_csv(f"{REPORTS_DIR}/invalid_samples.tsv", sep="\t", index=False)
                print(f"\nSaved invalid_samples.tsv with {len(invalid_all)} rows")
            else:
                print("\nNo invalid samples found.")
            print(f"\nReport saved to {REPORTS_DIR}/tsv_validation_report.txt")
        except Exception as e:
            print(f"WARNING: could not save report file: {e}")
    except Exception as e:
        print(f"ERROR during TSV validation: {e}")


## 10. Audio Validation

Validates every audio file referenced in the TSV: existence, duration, sample
rate, channels and silence. Skips if `df`/`AUDIO_DIR` are not ready.

In [ ]:
# Defensive defaults
try:
    df
except NameError:
    df = None
try:
    AUDIO_DIR
except NameError:
    AUDIO_DIR = None
try:
    AUDIO_COLUMN
except NameError:
    AUDIO_COLUMN = "audio"
try:
    MIN_DURATION_SEC
except NameError:
    MIN_DURATION_SEC = 0.5
try:
    MAX_DURATION_SEC
except NameError:
    MAX_DURATION_SEC = 30.0

# Initialise results shared with later sections
audio_results = []
total_duration = 0.0
sample_rates = {}
channels_count = {}
valid_count = 0
invalid_count = 0

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
elif not AUDIO_DIR or not os.path.exists(AUDIO_DIR):
    print(f"ERROR: AUDIO_DIR is not set or does not exist: {AUDIO_DIR}")
    print("Run Section 5 (Detect Dataset Paths) first.")
elif AUDIO_COLUMN not in df.columns:
    print(f"ERROR: Column '{AUDIO_COLUMN}' not found in TSV.")
else:
    try:
        print(f"Validating {len(df)} audio files...")
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Audio validation"):
            audio_path = row[AUDIO_COLUMN]

            def _add(status, reason, dur=None, sr=None, ch=None):
                audio_results.append({
                    "index": idx, "audio": audio_path, "status": status,
                    "reason": reason, "duration": dur, "sample_rate": sr, "channels": ch,
                })
                return None

            if pd.isna(audio_path) or str(audio_path).strip() == "":
                _add("invalid", "Empty audio path")
                invalid_count += 1
                continue

            full_path = os.path.join(AUDIO_DIR, str(audio_path).strip())
            if not os.path.exists(full_path):
                _add("invalid", "File not found")
                invalid_count += 1
                continue

            try:
                info = sf.info(full_path)
                duration = info.duration
                sr = info.samplerate
                ch = info.channels

                total_duration += duration
                sample_rates[sr] = sample_rates.get(sr, 0) + 1
                channels_count[ch] = channels_count.get(ch, 0) + 1

                reasons = []
                if duration < MIN_DURATION_SEC:
                    reasons.append(f"Too short ({duration:.2f}s)")
                if duration > MAX_DURATION_SEC:
                    reasons.append(f"Too long ({duration:.2f}s)")

                try:
                    y, _ = librosa.load(full_path, sr=None, mono=True)
                    if len(y) > 0 and np.max(np.abs(y)) < 0.001:
                        reasons.append("Silent audio")
                except Exception:
                    reasons.append("Cannot read audio data")

                if reasons:
                    _add("invalid", "; ".join(reasons), duration, sr, ch)
                    invalid_count += 1
                else:
                    _add("valid", "", duration, sr, ch)
                    valid_count += 1
            except Exception as e:
                _add("invalid", f"Cannot open: {str(e)}")
                invalid_count += 1

        total_hours = total_duration / 3600
        print(f"\n{'=' * 60}")
        print("AUDIO VALIDATION RESULTS")
        print(f"{'=' * 60}")
        print(f"Total samples:  {len(df)}")
        print(f"Valid samples:  {valid_count}")
        print(f"Invalid samples: {invalid_count}")
        print(f"Total duration: {total_hours:.2f} hours ({total_duration:.0f} seconds)")
        print("\nSample rates:")
        for sr, count in sorted(sample_rates.items()):
            print(f"  {sr} Hz: {count}")
        print("\nChannels:")
        for ch, count in sorted(channels_count.items()):
            label = "Mono" if ch == 1 else "Stereo" if ch == 2 else f"{ch}ch"
            print(f"  {label}: {count}")
    except Exception as e:
        print(f"ERROR during audio validation: {e}")

# Ensure total_hours exists for later cells
try:
    total_hours
except NameError:
    total_hours = 0.0


## 11. Save Audio Validation Report

Writes the audio validation summary to disk and lists invalid files.

In [ ]:
# Defensive defaults
for name, default in [
    ("audio_results", []), ("valid_count", 0), ("invalid_count", 0),
    ("total_duration", 0.0), ("sample_rates", {}), ("channels_count", {}),
    ("total_hours", 0.0), ("REPORTS_DIR", "/content/drive/MyDrive/khmer_tts/reports"),
]:
    try:
        globals()[name]
    except NameError:
        globals()[name] = default

if not audio_results:
    print("ERROR: No audio validation results. Run Section 10 (Audio Validation) first.")
else:
    try:
        audio_report_lines = []
        audio_report_lines.append("AUDIO VALIDATION REPORT")
        audio_report_lines.append("=" * 60)
        audio_report_lines.append(f"Total samples:  {len(audio_results)}")
        audio_report_lines.append(f"Valid samples:  {valid_count}")
        audio_report_lines.append(f"Invalid samples: {invalid_count}")
        audio_report_lines.append(f"Total duration: {total_hours:.2f} hours ({total_duration:.0f} seconds)")
        audio_report_lines.append("")
        audio_report_lines.append("Sample rates:")
        for sr, count in sorted(sample_rates.items()):
            audio_report_lines.append(f"  {sr} Hz: {count}")
        audio_report_lines.append("")
        audio_report_lines.append("Channels:")
        for ch, count in sorted(channels_count.items()):
            label = "Mono" if ch == 1 else "Stereo" if ch == 2 else f"{ch}ch"
            audio_report_lines.append(f"  {label}: {count}")
        audio_report_lines.append("")

        audio_report_text = "\n".join(audio_report_lines)
        os.makedirs(REPORTS_DIR, exist_ok=True)
        with open(f"{REPORTS_DIR}/audio_validation_report.txt", "w", encoding="utf-8") as f:
            f.write(audio_report_text)

        invalid_audio_df = pd.DataFrame([r for r in audio_results if r["status"] == "invalid"])
        if len(invalid_audio_df) > 0:
            invalid_audio_df.to_csv(f"{REPORTS_DIR}/invalid_audio.tsv", sep="\t", index=False)
            print(f"Saved invalid_audio.tsv with {len(invalid_audio_df)} entries")

        print(f"Audio report saved to {REPORTS_DIR}/audio_validation_report.txt")
    except Exception as e:
        print(f"ERROR saving audio report: {e}")


## 12. Khmer Text Validation

Validates Khmer transcripts: Unicode normalization, Khmer character coverage,
spaces, duplicates and punctuation. Skips if `df` is not loaded.

In [ ]:
import unicodedata
import re

try:
    df
except NameError:
    df = None
try:
    TEXT_COLUMN
except NameError:
    TEXT_COLUMN = "text"
try:
    REPORTS_DIR
except NameError:
    REPORTS_DIR = "/content/drive/MyDrive/khmer_tts/reports"

# Reset counters used by later sections
empty_count = 0
dup_text_count = 0
khmer_report_text = ""

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
elif TEXT_COLUMN not in df.columns:
    print(f"ERROR: Column '{TEXT_COLUMN}' not found.")
else:
    try:
        khmer_report = []
        khmer_report.append("KHMER TEXT VALIDATION REPORT")
        khmer_report.append("=" * 60)

        texts = df[TEXT_COLUMN].fillna("")

        empty_mask = texts.astype(str).str.strip() == ""
        empty_count = int(empty_mask.sum())
        khmer_report.append(f"Empty transcripts: {empty_count}")

        leading_trailing = int(texts.astype(str).str.contains(r"^\s|\s$", regex=True, na=False).sum())
        khmer_report.append(f"With leading/trailing spaces: {leading_trailing}")

        excessive_spaces = int(texts.astype(str).str.contains(r"\s{2,}", regex=True, na=False).sum())
        khmer_report.append(f"With excessive spaces: {excessive_spaces}")

        normalized = texts.apply(lambda x: unicodedata.normalize("NFC", str(x)))
        not_normalized = int((texts.astype(str) != normalized).sum())
        khmer_report.append(f"Not NFC-normalized: {not_normalized}")

        khmer_char_pattern = re.compile(r"[\u1780-\u17FF]")
        has_khmer = texts.astype(str).apply(lambda x: bool(khmer_char_pattern.search(str(x))))
        no_khmer = int((~has_khmer & (texts.astype(str).str.strip() != "")).sum())
        khmer_report.append(f"Rows without Khmer characters (non-empty): {no_khmer}")

        has_english = int(texts.astype(str).str.contains(r"[a-zA-Z]", regex=True, na=False).sum())
        khmer_report.append(f"Rows with English characters: {has_english}")

        has_numbers = int(texts.astype(str).str.contains(r"[0-9]", regex=True, na=False).sum())
        khmer_report.append(f"Rows with numbers: {has_numbers}")

        has_punct = int(texts.astype(str).str.contains(r"[.!?,;:\"\'\(\)\[\]{}]", regex=True, na=False).sum())
        khmer_report.append(f"Rows with Latin punctuation: {has_punct}")

        dup_text_count = int(texts.astype(str).duplicated().sum())
        khmer_report.append(f"Duplicate transcripts: {dup_text_count}")

        all_chars = set()
        for t in texts.astype(str):
            all_chars.update(str(t))
        khmer_report.append(f"Total unique characters: {len(all_chars)}")

        khmer_chars = sorted([c for c in all_chars if khmer_char_pattern.search(c)])
        khmer_report.append(f"Khmer characters found: {len(khmer_chars)}")

        khmer_report.append("")
        khmer_report_text = "\n".join(khmer_report)
        print(khmer_report_text)

        try:
            os.makedirs(REPORTS_DIR, exist_ok=True)
            with open(f"{REPORTS_DIR}/khmer_text_validation_report.txt", "w", encoding="utf-8") as f:
                f.write(khmer_report_text)
            print(f"\nReport saved to {REPORTS_DIR}/khmer_text_validation_report.txt")
        except Exception as e:
            print(f"WARNING: could not save Khmer report: {e}")
    except Exception as e:
        print(f"ERROR during Khmer text validation: {e}")


## 13. Create Validated Dataset Copy

Writes a cleaned `metadata_validated.tsv` (whitespace-normalized text/audio).
The original TSV is left untouched.

In [ ]:
try:
    df
except NameError:
    df = None
try:
    TEXT_COLUMN
except NameError:
    TEXT_COLUMN = "text"
try:
    AUDIO_COLUMN
except NameError:
    AUDIO_COLUMN = "audio"
try:
    PROCESSED_DIR
except NameError:
    PROCESSED_DIR = "/content/drive/MyDrive/khmer_tts/processed_dataset"
try:
    METADATA_TSV
except NameError:
    METADATA_TSV = None

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
    df_validated = None
else:
    try:
        df_validated = df.copy()

        if TEXT_COLUMN in df_validated.columns:
            df_validated[TEXT_COLUMN] = df_validated[TEXT_COLUMN].apply(
                lambda x: unicodedata.normalize("NFC", str(x).strip()) if pd.notna(x) else x
            )
            df_validated[TEXT_COLUMN] = df_validated[TEXT_COLUMN].apply(
                lambda x: re.sub(r"\s+", " ", str(x)).strip() if pd.notna(x) else x
            )

        if AUDIO_COLUMN in df_validated.columns:
            df_validated[AUDIO_COLUMN] = df_validated[AUDIO_COLUMN].apply(
                lambda x: str(x).strip() if pd.notna(x) else x
            )

        try:
            os.makedirs(PROCESSED_DIR, exist_ok=True)
            validated_path = f"{PROCESSED_DIR}/metadata_validated.tsv"
            df_validated.to_csv(validated_path, sep="\t", index=False)
            print(f"Saved validated dataset: {validated_path}")
            print(f"  Rows: {len(df_validated)}")
            print(f"  Columns: {list(df_validated.columns)}")
            print(f"\nOriginal TSV unchanged: {METADATA_TSV}")
        except Exception as e:
            print(f"ERROR saving validated copy: {e}")
    except Exception as e:
        print(f"ERROR creating validated copy: {e}")
        df_validated = None


## 14. Audio Preprocessing Test (10 files)

Tests resampling / mono conversion on up to 10 valid files. Originals are
never modified.

In [ ]:
import shutil

try:
    TEST_DIR
except NameError:
    TEST_DIR = "/content/drive/MyDrive/khmer_tts/processed_dataset/test"
try:
    AUDIO_DIR
except NameError:
    AUDIO_DIR = None
try:
    audio_results
except NameError:
    audio_results = []
try:
    TARGET_SAMPLE_RATE
except NameError:
    TARGET_SAMPLE_RATE = 22050

if not audio_results:
    print("ERROR: No audio validation results. Run Section 10 (Audio Validation) first.")
elif not AUDIO_DIR or not os.path.exists(AUDIO_DIR):
    print(f"ERROR: AUDIO_DIR is not set or does not exist: {AUDIO_DIR}")
else:
    try:
        os.makedirs(TEST_DIR, exist_ok=True)
        valid_audio = [r for r in audio_results if r.get("status") == "valid"]
        test_samples = valid_audio[:10]

        print(f"Testing preprocessing on {len(test_samples)} files...\n")
        preprocess_results = []
        for sample in test_samples:
            src_path = os.path.join(AUDIO_DIR, str(sample["audio"]))
            dst_path = os.path.join(TEST_DIR, f"processed_{os.path.basename(str(sample['audio']))}")
            try:
                y, orig_sr = librosa.load(src_path, sr=None, mono=False)
                if y.ndim > 1:
                    y = librosa.to_mono(y)
                if orig_sr != TARGET_SAMPLE_RATE:
                    y = librosa.resample(y, orig_sr=orig_sr, target_sr=TARGET_SAMPLE_RATE)
                    final_sr = TARGET_SAMPLE_RATE
                else:
                    final_sr = orig_sr
                peak = float(np.max(np.abs(y)))
                rms = float(np.sqrt(np.mean(y ** 2)))
                sf.write(dst_path, y, final_sr)
                preprocess_results.append({
                    "file": sample["audio"], "original_sr": orig_sr, "output_sr": final_sr,
                    "channels": 1, "peak": peak, "rms": rms,
                    "duration": float(len(y) / final_sr), "status": "ok",
                })
                print(f"  [OK] {sample['audio']}: {orig_sr}Hz -> {final_sr}Hz, peak={peak:.4f}, rms={rms:.4f}")
            except Exception as e:
                preprocess_results.append({"file": sample["audio"], "status": "error", "error": str(e)})
                print(f"  [ERROR] {sample['audio']}: {e}")

        print(f"\nProcessed files saved to: {TEST_DIR}")
        print("Originals untouched.")
    except Exception as e:
        print(f"ERROR during preprocessing test: {e}")


## 15. Dataset Statistics

Aggregates metrics into a JSON file saved under the reports directory.

In [ ]:
try:
    df
except NameError:
    df = None
for name, default in [
    ("valid_count", 0), ("invalid_count", 0), ("total_duration", 0.0),
    ("total_hours", 0.0), ("sample_rates", {}), ("channels_count", {}),
    ("empty_count", 0), ("dup_text_count", 0), ("METADATA_TSV", None),
    ("TARGET_SAMPLE_RATE", 22050), ("TARGET_CHANNELS", 1),
    ("REPORTS_DIR", "/content/drive/MyDrive/khmer_tts/reports"),
]:
    try:
        globals()[name]
    except NameError:
        globals()[name] = default

if df is None:
    print("ERROR: df not loaded. Run Section 7 (Load TSV) first.")
else:
    try:
        stats = {
            "total_samples": int(len(df)),
            "valid_audio_samples": int(valid_count),
            "invalid_audio_samples": int(invalid_count),
            "total_duration_seconds": float(total_duration),
            "total_duration_hours": float(total_hours),
            "sample_rates": {str(k): int(v) for k, v in sample_rates.items()},
            "channels": {str(k): int(v) for k, v in channels_count.items()},
            "empty_transcripts": int(empty_count) if TEXT_COLUMN in df.columns else None,
            "duplicate_transcripts": int(dup_text_count) if TEXT_COLUMN in df.columns else None,
            "columns": list(df.columns),
            "tsv_file": METADATA_TSV,
            "framework": "StyleTTS2",
            "license": "MIT",
            "target_sample_rate": int(TARGET_SAMPLE_RATE),
            "target_channels": int(TARGET_CHANNELS),
        }
        os.makedirs(REPORTS_DIR, exist_ok=True)
        with open(f"{REPORTS_DIR}/dataset_statistics.json", "w", encoding="utf-8") as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)

        print("Dataset Statistics:")
        print(json.dumps(stats, indent=2, ensure_ascii=False))
        print(f"\nSaved to {REPORTS_DIR}/dataset_statistics.json")
    except Exception as e:
        print(f"ERROR generating statistics: {e}")


## 16. Install StyleTTS2

Clones the StyleTTS2 repository and installs its dependencies.

In [ ]:
STYLETTS2_DIR = "/content/StyleTTS2"

try:
    if not os.path.exists(STYLETTS2_DIR):
        print("Cloning StyleTTS2 repository...")
        res = subprocess.run(
            ["git", "clone", "https://github.com/yl4579/StyleTTS2.git", STYLETTS2_DIR],
            capture_output=True, text=True,
        )
        if res.returncode == 0:
            print("Cloned StyleTTS2 repository.")
        else:
            print("WARNING: git clone failed.")
            print(res.stderr[-500:] if res.stderr else "")
    else:
        print("StyleTTS2 already cloned.")
except Exception as e:
    print(f"WARNING: clone step error: {e}")

try:
    req = os.path.join(STYLETTS2_DIR, "requirements.txt")
    if os.path.exists(req):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       check=False, capture_output=True, text=True)
    for extra in ["phonemizer", "inflect", "transformers", "accelerate"]:
        print(f"Installing {extra} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", extra],
                       check=False, capture_output=True, text=True)
    print("\nStyleTTS2 dependencies installed.")
except Exception as e:
    print(f"WARNING: dependency install error: {e}")


## 17. Test StyleTTS2 Model Load

Downloads base checkpoints and imports StyleTTS2 modules to confirm the
environment is ready for Phase 2.

In [ ]:
import sys

STYLETTS2_DIR = "/content/StyleTTS2"
sys.path.insert(0, STYLETTS2_DIR)

try:
    os.makedirs(os.path.join(STYLETTS2_DIR, "Models"), exist_ok=True)
    for fname, url in [
        ("LJSpeech.pth",
         "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LJSpeech.pth"),
        ("LJSpeech_config.yml",
         "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LJSpeech_config.yml"),
    ]:
        path = os.path.join(STYLETTS2_DIR, "Models", fname)
        if not os.path.exists(path):
            print(f"Downloading {fname} ...")
            subprocess.run(["wget", "-q", "-O", path, url], check=False,
                           capture_output=True, text=True)
        exists = os.path.exists(path)
        size = os.path.getsize(path) / (1024 * 1024) if exists else 0
        print(f"  [{'OK' if exists else 'MISSING'}] Models/{fname} ({size:.1f} MB)")
except Exception as e:
    print(f"WARNING: checkpoint download step error: {e}")

try:
    import torch
    device = "cuda" if (hasattr(torch, "cuda") and torch.cuda.is_available()) else "cpu"

    from Utils.XL import build_model
    from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

    print("StyleTTS2 modules loaded successfully.")
    print(f"Using device: {device}")

    if hasattr(torch, "cuda") and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        print(f"GPU memory before loading: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB")

    print("\nModel loading test: PASS")
    print("Environment ready for Phase 2 fine-tuning.")
except Exception as e:
    print(f"Model loading test result: {e}")
    print("Note: Full model loading will be done in Phase 2.")
    print("The key requirement is that StyleTTS2 code is cloned and dependencies are installed.")


## 18. GPU Memory Report

In [ ]:
try:
    import torch
    if hasattr(torch, "cuda") and torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        total_mem = getattr(props, "total_memory", 0) / 1024 ** 3
        print("GPU Memory Report:")
        print(f"  Total VRAM:     {total_mem:.1f} GB")
        print(f"  Allocated:      {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB")
        print(f"  Cached:         {torch.cuda.memory_reserved() / 1024 ** 3:.2f} GB")
        print(f"  Max allocated:  {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB")
    else:
        print("No GPU available for memory report.")
except Exception as e:
    print(f"GPU memory report skipped: {e}")


## 19. Convert to StyleTTS2 Format

Writes `train_list.txt` / `val_list.txt` in StyleTTS2's `audio_path|text` format.

In [ ]:
try:
    df_validated
except NameError:
    df_validated = None
try:
    AUDIO_COLUMN
except NameError:
    AUDIO_COLUMN = "audio"
try:
    TEXT_COLUMN
except NameError:
    TEXT_COLUMN = "text"
try:
    PROCESSED_DIR
except NameError:
    PROCESSED_DIR = "/content/drive/MyDrive/khmer_tts/processed_dataset"

if df_validated is None:
    print("ERROR: validated dataset not built. Run Section 13 first.")
elif AUDIO_COLUMN not in df_validated.columns or TEXT_COLUMN not in df_validated.columns:
    print("ERROR: Required columns not found in validated dataset.")
else:
    try:
        import numpy as _np
        lines = []
        for _, row in df_validated.iterrows():
            audio_file = str(row[AUDIO_COLUMN]).strip()
            text = str(row[TEXT_COLUMN]).strip()
            lines.append(f"{audio_file}|{text}")

        _np.random.seed(42)
        indices = _np.random.permutation(len(lines))
        split = int(0.9 * len(lines))
        train_lines = [lines[i] for i in indices[:split]]
        val_lines = [lines[i] for i in indices[split:]]

        training_meta_dir = f"{PROCESSED_DIR}/styletts2_format"
        os.makedirs(training_meta_dir, exist_ok=True)

        with open(f"{training_meta_dir}/train_list.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(train_lines))
        with open(f"{training_meta_dir}/val_list.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(val_lines))

        print("StyleTTS2 training format created:")
        print(f"  train_list.txt: {len(train_lines)} samples")
        print(f"  val_list.txt:   {len(val_lines)} samples")
        print("  Format: audio_path|text")
        print(f"  Location: {training_meta_dir}/")
        print("\nExample line:")
        print(f"  {train_lines[0] if train_lines else 'N/A'}")
    except Exception as e:
        print(f"ERROR converting to StyleTTS2 format: {e}")


## 20. Khmer G2P Research

**Phase 1 Finding:** No reliable Khmer G2P system exists as a pip-installable package.

### Options for Phase 2:
1. **Character-level training** — StyleTTS2 can train directly on Khmer Unicode characters.
2. **Custom Khmer G2P** — Build a rule-based grapheme-to-phoneme converter.
3. **MMS Phonemizer** — Meta's MMS for Khmer phoneme extraction.
4. **eSpeak-NG** — Partial Khmer support via the `kh` language code.

### Recommendation:
Start with **character-level training** (Option 1). Only build a custom G2P if quality is insufficient.

In [ ]:
import subprocess

try:
    result = subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True)
    print(f"eSpeak-NG: {result.stdout.strip()}")

    result = subprocess.run(
        ["espeak-ng", "-v", "kh", "--phonout=/dev/stdout", "\u17da\u179c\u17d2\u179a\u17b6\u1791"],
        capture_output=True, text=True,
    )
    if result.stdout.strip():
        print(f"Khmer phonemes: {result.stdout.strip()}")
    else:
        print("eSpeak-NG Khmer phonemization: Limited/empty output")
except FileNotFoundError:
    print("eSpeak-NG not installed. Install with: apt-get install espeak-ng")
except Exception as e:
    print(f"eSpeak-NG test: {e}")

print("\nKhmer G2P approach for Phase 2: Character-level training (recommended)")


## 21. Phase 1 Summary

In [ ]:
# Defensive defaults for summary counters
for name, default in [
    ("df", None), ("total_hours", 0.0), ("valid_count", 0), ("invalid_count", 0),
    ("sample_rates", {}), ("TARGET_CHANNELS", 1), ("METADATA_TSV", None),
    ("AUDIO_DIR", None), ("REPORTS_DIR", "/content/drive/MyDrive/khmer_tts/reports"),
    ("PROCESSED_DIR", "/content/drive/MyDrive/khmer_tts/processed_dataset"),
    ("TEST_DIR", "/content/drive/MyDrive/khmer_tts/processed_dataset/test"),
]:
    try:
        globals()[name]
    except NameError:
        globals()[name] = default

n_samples = len(df) if df is not None else 0
total_rows = f"{n_samples} samples" if df is not None else "N/A (df not loaded)"
duration = f"{total_hours:.2f} hours" if total_hours else "N/A"
sr_list = list(sample_rates.keys()) if sample_rates else ["unknown"]
channel_label = "Mono" if TARGET_CHANNELS == 1 else "Stereo"

print("=" * 60)
print("PHASE 1 COMPLETE - SUMMARY REPORT")
print("=" * 60)
print()
print(f"1. Dataset size:        {total_rows}")
print(f"2. Total audio:         {duration}")
print(f"3. Valid audio:         {valid_count}")
print(f"4. Invalid audio:       {invalid_count}")
print(f"5. Audio specs:         {sr_list} Hz, {channel_label}")
print(f"6. Selected model:      StyleTTS2")
print(f"7. Training framework:  StyleTTS2 (MIT License)")
print(f"8. GPU requirement:     NVIDIA T4 (15GB) minimum, better with more VRAM")
print(f"9. Khmer G2P:           Character-level (no G2P needed for Phase 2)")
print(f"10. Phase 2 tasks:      Fine-tune StyleTTS2 on Khmer dataset")
print()
print("Files created (if run in order):")
print(f"  - {REPORTS_DIR}/tsv_validation_report.txt")
print(f"  - {REPORTS_DIR}/audio_validation_report.txt")
print(f"  - {REPORTS_DIR}/invalid_samples.tsv")
print(f"  - {REPORTS_DIR}/invalid_audio.tsv")
print(f"  - {REPORTS_DIR}/dataset_statistics.json")
print(f"  - {REPORTS_DIR}/khmer_text_validation_report.txt")
print(f"  - {PROCESSED_DIR}/metadata_validated.tsv")
print(f"  - {PROCESSED_DIR}/styletts2_format/train_list.txt")
print(f"  - {PROCESSED_DIR}/styletts2_format/val_list.txt")
print(f"  - {TEST_DIR}/processed_* (test preprocessed files)")
print()
print("Original files UNTOUCHED:")
print(f"  - {METADATA_TSV}")
print(f"  - {AUDIO_DIR}/*")
print()
print("Ready for Phase 2: StyleTTS2 Fine-tuning on Khmer")
print("=" * 60)


## Phase 2 Roadmap

1. Fine-tune StyleTTS2 Stage 1 (diagnostic model) on Khmer data
2. Fine-tune StyleTTS2 Stage 2 (prosody predictor) on Khmer data
3. Train SLM adversarial component
4. Evaluate output quality with MOS testing
5. Build Khmer-specific G2P if character-level is insufficient
6. Create inference pipeline
7. Package model for deployment